# 01 — Qualidade dos Dados

Este notebook roda as mesmas validações de `src/validate_data.py` e mostra
o resultado de forma visual/exploratória, para documentar o processo de
qualidade de dados do Lobo Forecast AI.

Todos os dados são sintéticos e fictícios.

In [1]:
import sys

sys.path.insert(0, "../src")

import pandas as pd

from config import DATA_RAW_DIR

products = pd.read_csv(DATA_RAW_DIR / "products.csv")
calendar = pd.read_csv(DATA_RAW_DIR / "calendar.csv")
consumption = pd.read_csv(DATA_RAW_DIR / "consumption.csv")
inventory = pd.read_csv(DATA_RAW_DIR / "inventory_snapshot.csv")

print(f"products: {products.shape}")
print(f"calendar: {calendar.shape}")
print(f"consumption: {consumption.shape}")
print(f"inventory_snapshot: {inventory.shape}")

products: (30, 8)
calendar: (108, 6)
consumption: (5450, 5)
inventory_snapshot: (3120, 4)


## Checagens de integridade

In [2]:
checks = {
    "product_id único": not products["product_id"].duplicated().any(),
    "unit_cost sempre positivo": (products["unit_cost"] > 0).all(),
    "lead_time_days sempre positivo": (products["lead_time_days"] > 0).all(),
    "transaction_id único": not consumption["transaction_id"].duplicated().any(),
    "quantity sempre positiva": (consumption["quantity"] > 0).all(),
    "on_hand nunca negativo": (inventory["on_hand"] >= 0).all(),
    "in_transit nunca negativo": (inventory["in_transit"] >= 0).all(),
    "FK consumption -> products íntegra": consumption["product_id"].isin(products["product_id"]).all(),
    "FK inventory -> products íntegra": inventory["product_id"].isin(products["product_id"]).all(),
}

for check, passed in checks.items():
    status = "OK" if passed else "FALHOU"
    print(f"[{status}] {check}")

assert all(checks.values()), "Alguma checagem de qualidade falhou!"
print("\nTodas as checagens passaram.")

[OK] product_id único
[OK] unit_cost sempre positivo
[OK] lead_time_days sempre positivo
[OK] transaction_id único
[OK] quantity sempre positiva
[OK] on_hand nunca negativo
[OK] in_transit nunca negativo
[OK] FK consumption -> products íntegra
[OK] FK inventory -> products íntegra

Todas as checagens passaram.


## Distribuição de nulos e domínio de valores

In [3]:
print("Valores nulos por tabela:")
for name, df in [("products", products), ("calendar", calendar),
                  ("consumption", consumption), ("inventory", inventory)]:
    n_nulls = df.isna().sum().sum()
    print(f"  {name}: {n_nulls} valores nulos")

print("\nDomínio de criticality:", sorted(products["criticality"].unique()))
print("Domínio de category:", sorted(products["category"].unique()))
print("Domínio de work_center:", sorted(consumption["work_center"].unique()))

Valores nulos por tabela:
  products: 0 valores nulos
  calendar: 0 valores nulos
  consumption: 0 valores nulos
  inventory: 0 valores nulos

Domínio de criticality: ['HIGH', 'LOW', 'MEDIUM']
Domínio de category: ['ABRASIVOS', 'CORTE E USINAGEM', 'ELETRICA_BATERIAS', 'EPI_CONSUMIVEIS', 'SOLDAGEM']
Domínio de work_center: ['ELETRICA', 'FABRICACAO', 'MANUTENCAO', 'MECANICA', 'SOLDAGEM']


## Conclusão

Os dados sintéticos passam em todas as checagens estruturais e de domínio.
Isso confirma que o gerador (`src/generate_data.py`) e o validador
(`src/validate_data.py`) estão consistentes antes de qualquer análise ou
modelagem subsequente.